# ProteomeScout Update Pipeline

Generated by `generate_update_notebook.py` from the steps in `../README.md` — don't hand-edit
cells here, edit that script and re-run it instead.

Every run of this notebook writes a single timestamped log file to `../Data/new/logs/` recording
each action taken (files copied, species processed, resource IDs assigned, QC results).

## Manual steps before running this notebook
1. Copy `../config.example.json` to `../config.local.json` and set `data_root` to where external
   resource downloads (e.g. UniProt proteomes) live.
2. Have the path to the currently released ProteomeScout dataset directory (containing the
   combined `data.tsv` and `citations.tsv`) ready — everyone starts from these two files, not
   from data already split by species. The first code cell below copies them into `Data/current/`.


In [ ]:
import json
import os
import shutil
import subprocess
import sys
from datetime import datetime
from pathlib import Path

import pandas as pd

REPO_ROOT = Path("..").resolve()
PIPELINE_DIR = REPO_ROOT / "pipeline"
DATA_CURRENT = REPO_ROOT / "Data" / "current"
DATA_NEW = REPO_ROOT / "Data" / "new"
sys.path.insert(0, str(PIPELINE_DIR))

from run_log import start_log, log
from species_config import load_species_config
import translationTools

config_path = REPO_ROOT / "config.local.json"
if not config_path.exists():
    raise FileNotFoundError(
        f"Missing {config_path}. Copy config.example.json to config.local.json and set data_root."
    )
with config_path.open() as f:
    data_root = Path(json.load(f)["data_root"]).expanduser()

uniprot_data_dir = data_root / "Uniprot_Proteome"

LOG_FILE = start_log(DATA_NEW / "logs")
log(LOG_FILE, f"Started update cycle run. Log file: {LOG_FILE}")


def run(cmd):
    """Run a pipeline CLI command (list of str args), log it, and raise on failure."""
    log(LOG_FILE, f"Running: {' '.join(cmd)}")
    result = subprocess.run(cmd, cwd=PIPELINE_DIR)
    if result.returncode != 0:
        log(LOG_FILE, f"FAILED ({result.returncode}): {' '.join(cmd)}")
        raise RuntimeError(f"Command failed ({result.returncode}): {' '.join(cmd)}")
    log(LOG_FILE, f"Completed: {' '.join(cmd)}")


## Step -1: Stage the current released dataset into `Data/current/`

Enter the path to the directory holding the currently released combined `data.tsv` and
`citations.tsv`. Leave blank to skip if those are already copied into `Data/current/`.


In [ ]:
source_dir_input = input("Path to the current released ProteomeScout dataset directory (blank to skip): ").strip()

if source_dir_input:
    source_dir = Path(source_dir_input).expanduser()
    DATA_CURRENT.mkdir(parents=True, exist_ok=True)

    combined_source = source_dir / "data.tsv"
    shutil.copy2(combined_source, DATA_CURRENT / "data.tsv")
    log(LOG_FILE, f"Copied {combined_source} -> {DATA_CURRENT / 'data.tsv'}")

    citations_source = source_dir / "citations.tsv"
    citations_dest_dir = DATA_CURRENT / "ProteomeScout_Dataset"
    citations_dest_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(citations_source, citations_dest_dir / "citations.tsv")
    log(LOG_FILE, f"Copied {citations_source} -> {citations_dest_dir / 'citations.tsv'}")
else:
    log(LOG_FILE, "Skipped staging step; assuming Data/current/data.tsv is already populated.")


## Step 0: Split the combined dataset by species

Everyone starts from the single combined `Data/current/data.tsv` — this splits it into
`Data/current/<species>/data.tsv` (plus `species_other` for anything not in `species_config.json`).


In [ ]:
combined_file = DATA_CURRENT / "data.tsv"
if not combined_file.exists():
    raise FileNotFoundError(f"{combined_file} not found; stage the combined dataset in the previous cell first.")

run([
    "python", "split_species.py",
    "--input-file", str(combined_file),
    "--output-dir", str(DATA_CURRENT),
])
log(LOG_FILE, f"Split {combined_file} into per-species files under {DATA_CURRENT}")


## What resources are you updating today?

Enter a comma-separated list. Supported resources: `uniprot`, `psp`, `dbptm`.

> PSP is currently inactive — PhosphoSitePlus suspended academic license access as of
> September 2026. You can still select it, but the integration step will be skipped with a
> warning until access resumes.


In [ ]:
RESOURCE_CATALOG = {
    "uniprot": {
        "Name": f"UniProt Proteome Integration {datetime.now().strftime('%Y-%m-%d')}",
        "Citation": "UniProt Consortium. UniProt: the universal protein knowledgebase in 2021. Nucleic Acids Res. 2021 Jan 8;49(D1):D480-D489. doi: 10.1093/nar/gkaa1100. PMID: 33237286; PMCID: PMC7779014.",
        "Description": "Integration of UniProt proteome data into ProteomeScout for this update cycle.",
        "URL": "https://www.uniprot.org/",
        "PMID": "33237286",
    },
    "psp": {
        "Name": f"PhosphoSitePlus Integration {datetime.now().strftime('%Y-%m-%d')}",
        "Citation": "PhosphoSitePlus Consortium. PhosphoSitePlus: a comprehensive resource for investigating the structure and function of experimentally determined post-translational modifications in man and mouse. Nucleic Acids Res. 2014 Jan;42(Database issue):D431-7. doi: 10.1093/nar/gkt1249. PMID: 24234440; PMCID: PMC3965117.",
        "Description": "Integration of PhosphoSitePlus data into ProteomeScout for this update cycle.",
        "URL": "https://www.phosphosite.org/",
        "PMID": "24234440",
    },
    "dbptm": {
        "Name": f"dbPTM Integration {datetime.now().strftime('%Y-%m-%d')}",
        "Citation": "dbPTM consortium. dbPTM: an integrated resource for protein post-translational modifications.",
        "Description": "Integration of a normalized dbPTM dataset into ProteomeScout for this update cycle.",
        "URL": "https://biomics.lab.nycu.edu.tw/dbPTM/",
        "PMID": "",
    },
}

print(f"Available resources: {', '.join(RESOURCE_CATALOG)}")
selection = input("Which resources are you updating today? (comma-separated): ").strip()
RESOURCES_TO_RUN = [r.strip().lower() for r in selection.split(",") if r.strip()]

unknown = set(RESOURCES_TO_RUN) - set(RESOURCE_CATALOG)
if unknown:
    raise ValueError(f"Unknown resource(s) {unknown}; supported: {list(RESOURCE_CATALOG)}")

if "psp" in RESOURCES_TO_RUN:
    print("WARNING: PSP academic access is currently suspended; the PSP step below will be skipped.")

log(LOG_FILE, f"Resources selected for this cycle: {RESOURCES_TO_RUN}")


## Species configuration for this update cycle

Static per-species facts (taxid, proteome ID, scientific-name prefix) come from
`species_config.json`. Set which species run this cycle, and whether each is a full reference
proteome, below.


In [ ]:
SPECIES_IS_FULL_REFERENCE = {
    "human": True,
    "mouse": True,
    "rat": False,
    "cow": False,
    "fly": False,
    "yeast": False,
}

unknown_species = set(SPECIES_IS_FULL_REFERENCE) - set(load_species_config())
if unknown_species:
    raise ValueError(f"Add these species to species_config.json before running: {unknown_species}")

log(LOG_FILE, f"Species configured for this cycle: {SPECIES_IS_FULL_REFERENCE}")


## Step 1: Register resource IDs for the selected resources

Adds one row per selected resource to `citations.tsv` and writes the updated file to
`Data/new/ProteomeScout_Dataset/citations.tsv`.


In [ ]:
citations_file = DATA_CURRENT / "ProteomeScout_Dataset" / "citations.tsv"
citations_file_new = DATA_NEW / "ProteomeScout_Dataset" / "citations.tsv"
citations_file_new.parent.mkdir(parents=True, exist_ok=True)

RESOURCE_IDS = {}
current_citations_file = citations_file
for resource in RESOURCES_TO_RUN:
    metadata = RESOURCE_CATALOG[resource]
    resource_id, citations_df = translationTools.return_new_resource_id(
        current_citations_file, **metadata
    )
    citations_df.to_csv(citations_file_new, sep="\t", index=False)
    RESOURCE_IDS[resource] = resource_id
    current_citations_file = citations_file_new
    log(LOG_FILE, f"Registered resource '{resource}' as resource ID {resource_id}")

print(f"Resource IDs for this cycle: {RESOURCE_IDS}")


## Step 2: Update all records to current UniProt


In [ ]:
for species in SPECIES_IS_FULL_REFERENCE:
    run([
        "python", "update_proteomescout.py",
        "--data-dir", str(DATA_CURRENT),
        "--species", species,
        "--batch-size", "500",
    ])
    updated_file = DATA_CURRENT / species / "data.tsv.updated_clean"
    df = pd.read_csv(updated_file, sep="\t")
    qc_issues = translationTools.run_quality_control_mods(df)
    log(LOG_FILE, f"{species}: updated {len(df)} records, {len(qc_issues)} QC issues after update step")


## Step 3: Integrate UniProt proteome data

Skipped if `uniprot` wasn't selected above.


In [ ]:
if "uniprot" in RESOURCES_TO_RUN:
    for species, is_full_reference in SPECIES_IS_FULL_REFERENCE.items():
        run([
            "python", "uniprot_integration_pipeline.py",
            "--species", species,
            "--uniprot-data-dir", str(uniprot_data_dir),
            "--pscout-data-dir", str(DATA_CURRENT),
            "--resource-id", str(RESOURCE_IDS["uniprot"]),
            "--keep-records-not-in-pscout", str(is_full_reference),
            "--uniprot-swiss-nr", str(is_full_reference),
        ])
        integrated_file = DATA_CURRENT / species / "data.tsv.uniprot"
        df = pd.read_csv(integrated_file, sep="\t")
        qc_issues = translationTools.run_quality_control_mods(df)
        log(LOG_FILE, f"{species}: {len(df)} records, {len(qc_issues)} QC issues after UniProt integration")
else:
    log(LOG_FILE, "Skipped UniProt integration step (not selected this cycle).")


## Step 4: Integrate PhosphoSitePlus (PSP)

Skipped if `psp` wasn't selected above, or if PSP access is suspended.


In [ ]:
PSP_ACCESS_AVAILABLE = False  # flip to True once academic PSP access resumes

if "psp" in RESOURCES_TO_RUN and not PSP_ACCESS_AVAILABLE:
    log(LOG_FILE, "Skipped PSP integration: PSP was selected but academic access is suspended.")
elif "psp" in RESOURCES_TO_RUN:
    psp_data_file = input("Path to phosphositeplus_data.csv: ").strip()
    for species in SPECIES_IS_FULL_REFERENCE:
        run([
            "python", "psp_integration_pipeline.py",
            "--species", species,
            "--psp-data-file", psp_data_file,
            "--pscout-data-dir", str(DATA_CURRENT),
            "--resource-id", str(RESOURCE_IDS["psp"]),
            "--keep-records-not-in-pscout", "False",
            "--uniprot-swiss-nr", "False",
        ])
        integrated_file = DATA_CURRENT / species / "data.tsv.psp"
        df = pd.read_csv(integrated_file, sep="\t")
        qc_issues = translationTools.run_quality_control_mods(df)
        log(LOG_FILE, f"{species}: {len(df)} records, {len(qc_issues)} QC issues after PSP integration")
else:
    log(LOG_FILE, "Skipped PSP integration step (not selected this cycle).")


## Step 5: Integrate a normalized dbPTM dataset

Skipped if `dbptm` wasn't selected above.


In [ ]:
if "dbptm" in RESOURCES_TO_RUN:
    dbptm_dataset_file = input("Path to the normalized dbPTM dataset CSV: ").strip()
    for species in SPECIES_IS_FULL_REFERENCE:
        species_file = DATA_CURRENT / species / "data.tsv.uniprot"
        if not species_file.exists():
            species_file = DATA_CURRENT / species / "data.tsv.updated_clean"
        run([
            "python", "dbptm_integration_pipeline.py",
            "--dataset-file", dbptm_dataset_file,
            "--pscout-data-file", str(species_file),
            "--output-file", str(DATA_CURRENT / species / "data.tsv.dbptm"),
            "--resource-id", str(RESOURCE_IDS["dbptm"]),
        ])
        df = pd.read_csv(DATA_CURRENT / species / "data.tsv.dbptm", sep="\t")
        qc_issues = translationTools.run_quality_control_mods(df)
        log(LOG_FILE, f"{species}: {len(df)} records, {len(qc_issues)} QC issues after dbPTM integration")
else:
    log(LOG_FILE, "Skipped dbPTM integration step (not selected this cycle).")


## Step 6: Reintegrate species files into one dataset

Picks the most-integrated file available per species (dbPTM > PSP > UniProt > updated_clean).


In [ ]:
FILE_PRIORITY = ["data.tsv.dbptm", "data.tsv.psp", "data.tsv.uniprot", "data.tsv.updated_clean"]

df_dict = {}
for species in SPECIES_IS_FULL_REFERENCE:
    species_dir = DATA_CURRENT / species
    for filename in FILE_PRIORITY:
        candidate = species_dir / filename
        if candidate.exists():
            df_dict[species] = pd.read_csv(candidate, sep="\t")
            log(LOG_FILE, f"{species}: using {candidate} for final combined dataset")
            break
    else:
        raise FileNotFoundError(f"No integrated data file found for species '{species}' in {species_dir}")

combined_df = pd.concat(df_dict.values(), ignore_index=True)
DATA_NEW.mkdir(parents=True, exist_ok=True)
combined_output_file = DATA_NEW / "data.tsv"
combined_df.to_csv(combined_output_file, sep="\t", index=False)
log(LOG_FILE, f"Wrote combined dataset ({len(combined_df)} records) to {combined_output_file}")


## Step 7: Final QC


In [ ]:
qc_report_file = DATA_NEW / "qc_report.json"
run([
    "python", "qc_integrated_data.py",
    "--input-file", str(DATA_NEW / "data.tsv"),
    "--report-json", str(qc_report_file),
])
log(LOG_FILE, f"QC complete. Report: {qc_report_file}")
log(LOG_FILE, "Update cycle finished. Once satisfied, promote Data/new/ to Data/current/ for the next cycle.")
